In [1]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.tests.nodes.reformulate import send_reformulate_requests, save_reformualte_responses, reformulate_tests
from app.utils.db import save_results_on_cosmos

In [3]:
file_list = [
  # './app/data/raw/tramites.xlsx',
  # './app/data/raw/accesibilidad.xlsx',
  # './app/data/raw/descubrir.xlsx', 
  # './app/data/raw/solicitudes.xlsx',
  # './app/data/raw/organigrama.xlsx'
  './app/data/raw/rapido.xlsx'
]

test_config = {
  'GENERAL_TESTS': True,
  'TIMINGS': {'test': True, 'report': True},
  'TOKENS': {'test': False, 'report': False},
  'FOUNDRYS': {'test': False, 'report': False},
  'TRIAGE': {'test': True, 'report': True},
  'ROUTER': {'test': True, 'report': True},
  'GROUNDING': {'test': True, 'report': True},
  'SAVE_RESULTS': True,
  'PATH': './app/data/processed/reports/report_5chat',
  
  'REFORMULATE': {'test': False, 'report': False}
}   




In [4]:
if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)


with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [5]:
# TEST GENERALES
if test_config.get('GENERAL_TESTS', False):
  responses = client.query_batch(df['user_input'],df['reference'])
  save_responses_in_json, response_file_path = client.save_api_responses(responses)
  test_timestamps['general_tests'] = str(response_file_path)
  

# SOLO REFORMULATE
if test_config.get('REFORMULATE', False).get('test', False):
  reformulate_dataset = load_test_cases('./app/data/raw/reformulate.xlsx')
  reformulate_results = send_reformulate_requests(config= config_data, dataset=reformulate_dataset)
  generate_reformulate_json, reformulate_timestamp = save_reformualte_responses(reformulate_results)
  test_timestamps['reformulate_test'] = reformulate_timestamp
  reformulate_results = reformulate_tests(reformulate_results)

Processing queries: 100%|██████████| 100/100 [20:23<00:00, 12.24s/it]


In [7]:
import json

response_file_path = './app/data/processed/outcomes/outcome_20260428-171719.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)
test_timestamps['general_tests'] = 'outcome_20260428-171719.json'

In [6]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
  )

print(results)

{'timestamp': '20260430-112206', 'nodes': {'triage': {'positives': 88, 'total': 100, 'result': 88.0}, 'router': {'positives': 82, 'total': 88, 'result': 93.18}, 'grounding': {'positives': 79, 'total': 86, 'result': 91.86}}, 'timings': {'reformulate': {'prom': 1.428, 'med': 1.186, 'p90': 2.189, 'p95': 3.014, 'p99': 3.925, 'quantity': 100}, 'triage': {'prom': 1.331, 'med': 1.256, 'p90': 1.748, 'p95': 2.061, 'p99': 2.871, 'quantity': 100}, 'router': {'prom': 0.68, 'med': 0.407, 'p90': 1.719, 'p95': 1.873, 'p99': 2.274, 'quantity': 100}, 'ag_call': {'prom': 6.067, 'med': 5.005, 'p90': 8.434, 'p95': 9.795, 'p99': 20.323, 'quantity': 86}, 'personality': {'prom': 3.138, 'med': 3.185, 'p90': 4.316, 'p95': 5.077, 'p99': 6.223, 'quantity': 86}, 'grounding': {'prom': 1.585, 'med': 1.449, 'p90': 2.072, 'p95': 2.779, 'p99': 4.102, 'quantity': 86}, 'retriever': {'prom': 0.586, 'med': 0.513, 'p90': 0.812, 'p95': 1.01, 'p99': 1.707, 'quantity': 86}, 'ret_embeddings': {'prom': 0.32, 'med': 0.261, 'p90'

In [ ]:
save_results_on_cosmos(results)

In [11]:
agents_timings = {
  'tramites': [],
  'accesibilidad': [],
  'descubrir': []
}

for item in responses:
    
    list_of_agents = ['tramites', 'accesibilidad', 'descubrir']

    if item is None or 'ok' in item:
      continue

    # if 'router' in item.get('partial_answers', {}):     
    route = item.get('partial_answers', {}).get('router', {}).get('route', '')

    node_metadata = item.get("node_metadata", {})
    t_response_time = item.get('response_time', 0)

    if route:
      if route == 'tramites':
        agents_timings['tramites'].append(t_response_time)
      elif route == 'accesibilidad':
        agents_timings['accesibilidad'].append(t_response_time)
      elif route == 'descubrir':
        agents_timings['descubrir'].append(t_response_time)

average_results = {
  'tramites': round(sum(agents_timings['tramites']) / len(agents_timings['tramites']),2),
  'accesibilidad': round(sum(agents_timings['accesibilidad']) / len(agents_timings['accesibilidad']), 2),
  'descubrir': round(sum(agents_timings['descubrir']) / len(agents_timings['descubrir']), 2),
}
print(average_results)

{'tramites': 12.28, 'accesibilidad': 12.03, 'descubrir': 9.76}
